# Tutorial 04b: Logistic Regression

Mathematics for Machine Learning

Thomas Manke  
2026-11-19

## Setup

Run the setup below to load the
<a href="https://archive.ics.uci.edu/dataset/45/heart+disease"
target="_blank" rel="noopener">Cleveland Heart Disease Dataset</a> from
the UCI Machine Learning Repository (Detrano et al., 1989) using
`fetch_ucirepo`.

In [1]:
import sys
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    log_loss,
    accuracy_score
)

sns.set_theme(context="notebook", style="whitegrid")
FIGSIZE = (7, 4)
RANDOM_STATE = 42

# Load Cleveland Heart Disease Dataset via fetch_ucirepo (UCI ID: 45)
from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=45)
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)
# Optional: remove ordinal categorical variables (e.g., slope) to avoid complex encodings
df = df.drop(columns=['slope'])  # Comment out to retain slope
print(f"Loaded via fetch_ucirepo (ID: 45) | Variable Types: {dataset.variables['type'].value_counts().to_dict()}")

print(f"Raw dataset shape: {df.shape}")
df.head()

------------------------------------------------------------------------

## Task 1: Data Preparation

- **Data Cleaning & Target Encoding:** Inspect the raw dataset. Remove
  any missing values (NAs) if present. Encode the target variable such
  that $y = 1$ (Heart Disease present, `num > 0`) and $y = 0$ (Healthy,
  `num = 0`).
- **Categorical Feature Encoding:** Apply one-hot encoding with
  `drop_first=True` to nominal categorical variables (`cp`, `restecg`,
  `thal`), keep count variables (`ca`) as integers, and retain binary
  indicators (`sex`, `fbs`, `exang`) in $0/1$ form. Define the feature
  matrix $\mathbf{X}$ and target vector $y$.
- **Class Imbalance:** Investigate the class distribution and determine
  the disease prevalence $\bar{p}$.
- **Stratified Split & Standardization:** Create a stratified train/test
  split (80% train, 20% test) and standardize the features using
  `StandardScaler`.

In [3]:
# Stratified train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# Standardize features
scaler = StandardScaler()
# Fit and transform training data
X_train_scaled = scaler.fit_transform(X_train)
# Only transform test data
X_test_scaled = scaler.transform(X_test)

df_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols)
df_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols)

print(f"Training set: {X_train_scaled.shape[0]} patients")
print(f"Test set    : {X_test_scaled.shape[0]} patients")

------------------------------------------------------------------------

## Task 2: Baseline Model

- **Prevalence, Odds & Null Deviance:** Using the training data, define
  a baseline model with no predictors and only use the prevalence
  $\bar{p}$ for all observations. Calculate the odds, log-odds, and the
  null deviance of the training data
  $$D_0 = -2 \sum_{i=1}^n [y_i \ln \bar{p} + (1 - y_i)\ln(1 - \bar{p})]$$
- **Intercept-Only Logistic Regression:** Fit a logistic regression
  model with only an intercept term, extract the fitted parameter
  $\hat{\theta}_0$, and compare to the baseline log-odds.
- **Baseline Plot:** Plot the observed target values and the null model
  prediction $\hat{p} = \bar{p}$ for all samples (you may sort by sample
  index).

------------------------------------------------------------------------

## Task 3: Univariate Logistic Regression

- **Univariate Models & Ranking:** Conduct univariate logistic
  regressions for each feature $j$ independently. Compute the residual
  deviance $D(x_j)$ and the deviance drop relative to the null model,
  $\Delta D_j = D_0 - D(x_j)$. Display the ranking table.
- **Continuous Feature (Model 1):** For the top-ranked continuous
  feature (`oldpeak`: exercise ST depression), plot both the predicted
  probability $\hat{p}(x)$ and the log-odds $z(x)$.
- **Parameter Interpretation & Threshold:** Interpret the slope
  parameter $\theta_1$ and determine the decision threshold $x^*$ where
  $\hat{p} = 0.5$.

------------------------------------------------------------------------

## Task 4: Bivariate Logistic Regression

- **Bivariate Model (Model 2):** Fit a bivariate logistic regression
  combining the two most predictive continuous features from exercise
  stress testing: $x_1$ (`oldpeak`, ST depression) and $x_2$ (`thalach`,
  maximum heart rate achieved).
- **2D Probability Gradient & Decision Boundary:** Plot the predicted
  probability $\hat{p}$ across the 2D grid $(x_1, x_2)$ as a smooth
  contour gradient and plot the observed data points colored distinctly
  by disease status (Healthy vs. Heart Disease).
- **Deviance Progression:** Calculate the residual deviance $D_2$ and
  confirm that $D_2 < D_1 < D_0$.

------------------------------------------------------------------------

## Task 5: Loss Landscape

- **Scan Parameter:** Fix the intercept $\theta_0 = \hat{\theta}_{0}$
  from Model 1 and scan the slope parameter $\theta_1$ across a broad
  range around the best prediction,
  $\theta_1 \in [-5 |\hat{\theta}_1|, +5 |\hat{\theta}_1|]$.
- **Loss Curves Comparison:** Compute and plot the Binary Cross-Entropy
  (BCE) and Mean Squared Error (MSE) for predictions
  $\hat{y}_i = \sigma(x_i^T \theta)$. What are the key differences in
  the asymptotic behavior of BCE vs. MSE?

------------------------------------------------------------------------

## Task 6: Multivariate Logistic Regression

- **Full Model Fitting:** Fit a multivariable logistic regression using
  all encoded standardized features with default parameters.
- **Deviance Comparison:** Compare the residual deviance $D$, and
  deviance drop $\Delta D$ for the four models (Model 0: Null, Model 1:
  Univariate `oldpeak`, Model 2: Bivariate `oldpeak` + `thalach`, Model
  3: Full).

------------------------------------------------------------------------

## Task 7: Model Evaluation

- **Evaluation on the Test Set:** Evaluate the fitted full model
  (Model 3) on the unseen **test set** ($\mathbf{X}_{\text{test}}$,
  $\mathbf{y}_{\text{test}}$) using a decision threshold of $t = 0.5$.
  Calculate and visualize the Confusion Matrix using scikit-learn, and
  calculate Sensitivity (Recall), Specificity, Precision, and Accuracy.
- **Diagnostic Curves:** Plot the ROC and Precision-Recall curves for
  the test set predictions, including the AUC and Average Precision (AP)
  scores.

------------------------------------------------------------------------

## Task 8: Optional Regularization

The default implementation of `LogisticRegression()` in scikit-learn
applies $L_2$ **Ridge** regularization with a default penalty strength
of $C=1.0$. Explore the impact of this parameter on model performance.
An alternative regularization approach is $L_1$ **Lasso**, which can
drive redundant collinear features to exact zero, yielding a sparser
model. You can explore this with parameters `C`, `l1_ratio=1` and
`solver='saga'`. How many features remain active (non-zero) under Lasso?
Which are removed?

------------------------------------------------------------------------

## Checkpoint and Review Questions

- **Question 1 (Training vs. Test Data & Preprocessing Leakage):** Why
  must ML models be evaluated on an independent test set? Why is it
  critical that feature scaling (e.g. `StandardScaler`) is fitted
  strictly on the training partition?
- **Question 2 (Parameter Interpretation & Odds Ratios):** In logistic
  regression, how is a fitted slope parameter $\theta_j$ interpreted in
  terms of the Odds Ratio ($\text{OR}_j = \exp(\theta_j)$)? What does it
  mean for the odds if $\theta_j > 0$, $\theta_j = 0$, or
  $\theta_j < 0$?
- **Question 3 (Confusion Matrix & ROC):** In medical diagnostics, why
  is overall classification accuracy often misleading? Name two
  complementary metrics from the confusion matrix that distinguish
  between error types. Explain what the ROC curve shows. Why is AUC a
  useful summary statistic?